# Deep Gated RNN — Sentiment Classification on IMDB (Kaggle)

A **deep, bidirectional, residual GRU** trained from scratch on the
[IMDB 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) dataset.

**Architecture**
1. Embedding + dropout, projected to the model width
2. N stacked *bidirectional GRU blocks*, each with **residual connection + LayerNorm + dropout**
3. **Masked attention pooling** + **masked max pooling** (concatenated)
4. MLP classifier head

Run on a GPU runtime (Colab / Kaggle). Kaggle API credentials are needed for `kagglehub` outside Kaggle.

In [ ]:
!pip install -q kagglehub torch pandas numpy scikit-learn matplotlib seaborn tqdm

In [ ]:
import os, re, json, math, random, time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Configuration

In [ ]:
CFG = dict(
    max_vocab   = 30_000,
    min_freq    = 2,
    max_len     = 400,
    embed_dim   = 256,
    hidden_dim  = 256,     # per direction
    num_layers  = 4,       # depth of the gated stack
    dropout     = 0.3,
    batch_size  = 64,
    epochs      = 12,
    lr          = 2e-3,
    weight_decay= 1e-2,
    grad_clip   = 1.0,
    patience    = 3,       # early stopping
)
CFG

## 2. Download the dataset from Kaggle

In [ ]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
csv_path = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(csv_path)
df["label"] = (df["sentiment"] == "positive").astype(int)
print(df.shape); df.head()

In [ ]:
sns.countplot(x="sentiment", data=df); plt.title("Class balance"); plt.show()
df["n_words"] = df["review"].str.split().str.len()
print(df["n_words"].describe())

## 3. Preprocessing & vocabulary

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9']+|[!?.,]")

def clean(text):
    text = re.sub(r"<br\s*/?>", " ", text.lower())
    return TOKEN_RE.findall(text)

df["tokens"] = [clean(t) for t in tqdm(df["review"], desc="Tokenizing")]

train_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df["label"], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

counter = Counter(tok for toks in train_df["tokens"] for tok in toks)
itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common(CFG["max_vocab"]) if c >= CFG["min_freq"]]
stoi = {w: i for i, w in enumerate(itos)}
PAD, UNK = 0, 1
print("Vocab size:", len(itos))

def encode(tokens):
    ids = [stoi.get(t, UNK) for t in tokens[: CFG["max_len"]]]
    return ids if ids else [UNK]

In [ ]:
class TextDS(Dataset):
    def __init__(self, frame):
        self.x = [encode(t) for t in frame["tokens"]]
        self.y = frame["label"].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

def collate(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    padded = torch.full((len(xs), int(lengths.max())), PAD, dtype=torch.long)
    for i, x in enumerate(xs):
        padded[i, : len(x)] = torch.tensor(x)
    return padded, lengths, torch.tensor(ys, dtype=torch.float)

def make_loader(frame, shuffle):
    return DataLoader(TextDS(frame), batch_size=CFG["batch_size"], shuffle=shuffle,
                      collate_fn=collate, num_workers=2, pin_memory=True)

train_dl, val_dl, test_dl = make_loader(train_df, True), make_loader(val_df, False), make_loader(test_df, False)

## 4. Model: Deep Residual Bidirectional GRU

Each **GatedBlock** is `x -> BiGRU -> Dropout -> + x -> LayerNorm`.
Residual paths keep gradients healthy through the deep stack; LayerNorm stabilises training.
Packed sequences make the GRU ignore padding.

In [ ]:
class GatedBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        assert dim % 2 == 0
        self.rnn  = nn.GRU(dim, dim // 2, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        return self.norm(x + self.drop(out))


class DeepGatedRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx=0):
        super().__init__()
        dim = hidden_dim * 2                      # width of the residual stream
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_drop = nn.Dropout(dropout)
        self.proj  = nn.Linear(embed_dim, dim)
        self.blocks = nn.ModuleList([GatedBlock(dim, dropout) for _ in range(num_layers)])
        self.attn  = nn.Linear(dim, 1)
        self.head  = nn.Sequential(
            nn.Linear(dim * 2, dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim, 1)
        )

    def forward(self, ids, lengths):
        mask = ids != PAD                                          # (B, T)
        x = self.proj(self.embed_drop(self.embed(ids)))
        for blk in self.blocks:
            x = blk(x, lengths)

        scores = self.attn(x).squeeze(-1).masked_fill(~mask, -1e9)
        w = torch.softmax(scores, dim=1).unsqueeze(-1)
        attn_pool = (w * x).sum(1)
        max_pool  = x.masked_fill(~mask.unsqueeze(-1), -1e9).max(1).values
        return self.head(torch.cat([attn_pool, max_pool], dim=-1)).squeeze(-1)


model = DeepGatedRNN(len(itos), CFG["embed_dim"], CFG["hidden_dim"], CFG["num_layers"], CFG["dropout"]).to(device)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Training

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
steps = CFG["epochs"] * len(train_dl)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG["lr"], total_steps=steps, pct_start=0.1)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for ids, lengths, y in tqdm(loader, leave=False):
            ids, y = ids.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(ids, lengths)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                scaler.step(optimizer); scaler.update(); scheduler.step()
            total_loss += loss.item() * y.size(0)
            correct += ((logits > 0).float() == y).sum().item()
            n += y.size(0)
    return total_loss / n, correct / n

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val, bad_epochs = 0.0, 0
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    tl, ta = run_epoch(train_dl, True)
    vl, va = run_epoch(val_dl, False)
    for k, v in zip(history, (tl, vl, ta, va)): history[k].append(v)
    print(f"Epoch {epoch:02d} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} | {time.time()-t0:.0f}s")
    if va > best_val:
        best_val, bad_epochs = va, 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= CFG["patience"]:
            print("Early stopping."); break

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["train_acc"],  label="train"); ax[1].plot(history["val_acc"],  label="val"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

## 6. Evaluation on the held-out test set

In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()
preds, labels = [], []
with torch.no_grad():
    for ids, lengths, y in test_dl:
        logits = model(ids.to(device), lengths)
        preds += (logits > 0).long().cpu().tolist()
        labels += y.long().tolist()

print(classification_report(labels, preds, target_names=["negative", "positive"], digits=4))
cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix")
plt.savefig("confusion_matrix.png", dpi=150); plt.show()

## 7. Inference helper & saving artifacts

In [ ]:
@torch.no_grad()
def predict(text):
    model.eval()
    ids = torch.tensor([encode(clean(text))], device=device)
    lengths = torch.tensor([ids.size(1)])
    p = torch.sigmoid(model(ids, lengths)).item()
    return {"label": "positive" if p > 0.5 else "negative", "confidence": round(max(p, 1 - p), 4)}

for s in ["An absolute masterpiece, I was hooked from the first minute.",
          "Dull, predictable and far too long. I want my time back."]:
    print(s, "->", predict(s))

with open("vocab.json", "w") as f: json.dump(itos, f)
with open("config.json", "w") as f: json.dump(CFG, f, indent=2)